Import modules we need

In [1]:
from utils import *
from google.transit import gtfs_realtime_pb2
ROOT = Path("../")
ROOT.resolve()

PosixPath('/Users/lukestrange/Code/bus-tracking')

Unzip the GTFS timetable files

In [2]:
GTFS_file = ROOT / "18SepGB_GTFS_Timetables_Downloaded/itm_yorkshire_gtfs.zip"
DIR = os.path.dirname(GTFS_file)
# Specify the path to your zip file
zip_file_path = ROOT / "18SepGB_GTFS_Timetables_Downloaded/itm_yorkshire_gtfs.zip"

# Define the directory where you want to extract files temporarily (optional)
extract_dir = ROOT / "18SepGB_GTFS_Timetables_Downloaded/yorkshire"

# Open the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip:
    zip.extractall(extract_dir)
    file_paths = zip.namelist()
    
    # Go through each file in the zip archive
    for file_name in file_paths:
        # Construct the full file path of the extracted file
        full_file_path = os.path.join(extract_dir, file_name)

Make dictionaries based on the GTFS files

In [3]:
for file in os.listdir(extract_dir):
    fp = os.path.join(extract_dir, file)
    data = pd.read_csv(fp, low_memory=False)
    if file == 'agency.txt':
        agencies = data[['agency_id', 'agency_noc']]
    if file == 'stops.txt':
        stops = data[['stop_id', 'stop_lat', 'stop_lon']]
    if file == 'stop_times.txt':
        stop_times = data[['trip_id', 'stop_id']]
    if file == 'trips.txt':
        trips = data[['trip_id', 'route_id']]
    if file == 'calendar.txt':
        calendar = data
    if file == 'calendar_dates.txt':
        calendar_dates = data
    if file == 'routes.txt':
        routes = data[['agency_id', 'route_id', 'route_short_name', 'route_type']]

# Make the dataframes to be used to make dicts
# ------ Don't need these two below for now.
agency2route = agencies.merge(routes, on='agency_id').set_index('agency_id')
route2trip = routes.merge(trips, on='route_id').set_index('route_id')
trip2stoptimes = trips.merge(stop_times, on='trip_id')[['trip_id', 'stop_id']].groupby('trip_id').agg(list)
stoptimes2stops = stop_times.merge(stops, on='stop_id').set_index('stop_id')

# Covnert DFs to dictionaries
#------- Don't need these two for now
AgencyNOC2AgencyIDDict = agency2route['agency_noc'].drop_duplicates().to_dict()
RouteID2TripIDDict = route2trip['trip_id'].to_dict()
Trip2StopIDDict = trip2stoptimes['stop_id'].drop_duplicates().to_dict()
StopID2StopLocDict = stoptimes2stops[['stop_lat', 'stop_lon']].drop_duplicates().to_dict(orient='index')

In [4]:
# stopID2stoploc_df = pd.DataFrame.from_dict(StopID2StopLocDict, orient='index')

Get the paths of the GTFS-RT files

In [5]:
# Now load the GTFS-RT file
gtfs_rt_dir = ROOT / 'extracted_19SepGB_BusLocations_GTFSRT' 

# Create an empty list to store file paths
gtfs_rt_file_paths = []
# Walk through the directory
for root, dirs, files in os.walk(gtfs_rt_dir):
    for file in files:
        # Get the full path of the file and append it to the list
        full_path = os.path.abspath(os.path.join(root, file))
        gtfs_rt_file_paths.append(full_path)

Load the feed entities into an array to loop through later

In [6]:
feed = gtfs_realtime_pb2.FeedMessage()
entities = []
for gtfsrt in gtfs_rt_file_paths:
    # Read the GTFS-RT file
    # For now only going to do 1hr - 8-9am.
    if 'h' in str(gtfsrt):
        with open(gtfsrt, 'rb') as file:
            feed.ParseFromString(file.read())
            for entity in feed.entity:
                entities.append(entity)

Getting the real-time locations of Buses.

In [7]:
BusDetailsBag = []
# count = 0
# t0 = time.time()
# For the moment assuming that `vehicle` is passed. there are other mutually exclusive 
# options: 'trip_update', 'alert', and 'shape'. See the DOCS https://gtfs.org/documentation/realtime/reference/#message-feedentity
for entity in entities:
    BD = BusDetail()
    # These are the two main parts of entity
    vehicle = entity.vehicle
    # # These are sub parts
    trip = vehicle.trip
    pos = vehicle.position
    # These are individual values
    BD.feed_uid = entity.id
    BD.trip_id = trip.trip_id
    BD.route_id = trip.route_id
    BD.lat = pos.latitude
    BD.lon = pos.longitude
    BD.bearing = pos.bearing
    BD.ts = vehicle.timestamp
    BD.v_id = vehicle.vehicle.id
    BD.occupancy_status = vehicle.occupancy_status
    BD.current_stop_sequence = vehicle.current_stop_sequence
    BD.current_status = vehicle.current_status

    if BD.trip_id:
        stops_on_route = Trip2StopIDDict.get(BD.trip_id)
        # If stops isn't none - i.e. if this trip ID is part of the WY timetable
        if stops_on_route:
            stopID2locs_this_route = stopID2stoploc_df[stopID2stoploc_df.index.isin(stops_on_route)].copy()
            stopID2locs_this_route['distance'] = stopID2locs_this_route.apply(lambda row: haversine(BD.lat, BD.lon, row['stop_lat'], row['stop_lon']), axis=1)
            closest_stop = stopID2locs_this_route.loc[stopID2locs_this_route['distance'].idxmin()]
            BD.NearestStopOnRoute = closest_stop.name
            BD.NearestStopDistance = closest_stop.distance * 1e3 #convert to m
            if BD.NearestStopOnRoute != None and BD.NearestStopDistance < 200:
                BusDetailsBag.append(BD)
    # if count % 100000 == 0:
    #     t1 = time.time()
        # print('Time elapsed:', round(t1-t0, 3), 's')
        # print(f"{count} of {len(entities)} entities parsed.")
    # count +=1

NameError: name 'stopID2stoploc_df' is not defined

Now delete duplicate stops (this also gets rid of long periods where the vehicle is waiting)

Sort the frame by timestamp descending, then keep FIRST of duplicate rows.

In [67]:
data = [{'trip_id': bus.trip_id, "route_id": bus.route_id, 'timestamp': bus.ts, 'nearest_stop_id': bus.NearestStopOnRoute, 'distance': bus.NearestStopDistance} for bus in BusDetailsBag]
df = pd.DataFrame(data)
df['human_time'] = pd.to_datetime(df['timestamp'], unit='s')
# Timezone is currently BST =  UTC + 1, so need to add 1 hour.
df['time_only'] = (df['human_time'] + pd.Timedelta(hours=1)).dt.strftime('%H:%M:%S')
# # df = df.groupby(['trip_id', 'route_id'])
df.sort_values(by=['trip_id', 'route_id', 'timestamp'], ascending=True, inplace=True)

# # # Weird rows where they aren't the right date. Binning them
df = df[df.human_time.dt.date == pd.to_datetime('2024-09-19').date()]
# only remove duplictaes that have same stop_id, route_id and nearest_stop_id
df.drop_duplicates(subset=['trip_id', 'route_id', 'nearest_stop_id'], keep='first', inplace=True)
FilteredOrderedBusLocations = df

Reading in the original GTFS timetable data. We'll use this to pull all the info about the routes based on the trips we were actually able to match buses for.

In [68]:
for file in os.listdir(extract_dir):
    fp = os.path.join(extract_dir, file)
    data = pd.read_csv(fp, low_memory=False)
    if file == 'agency.txt':
        agencies = data
    if file == 'stops.txt':
        stops = data
    if file == 'stop_times.txt':
        stop_times = data
    if file == 'trips.txt':
        trips = data
    if file == 'calendar.txt':
        calendar = data
    if file == 'calendar_dates.txt':
        calendar_dates = data
    if file == 'routes.txt':
        routes = data
    if file == 'shapes.txt':
        shapes = data

Now write this timetable as GTFS. This includes:
- agency.txt
- calendar.txt
- calendar_dates.txt
- routes.txt
- stop_times.txt
- stops.txt 
- trips.txt
- shapes.txt

In [69]:
RealRouteIDs = FilteredOrderedBusLocations['route_id'].astype(int)
RealRoutes = routes[routes['route_id'].isin(RealRouteIDs)]
RealRoutes.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/routes.txt", index=False)
RealRoutes

,route_id,agency_id,route_short_name,route_long_name,route_type
256,6554,OP12112,85,NaN,3
257,12443,OP926,AL4,NaN,3
258,103593,OP4522,C81,NaN,3
259,12614,OP933,X11,NaN,3
260,12807,OP318,K7,NaN,3
...,...,...,...,...,...
1265,42705,OP229,X4A,NaN,3
1266,19856,OP8911,86A,NaN,3
1269,94884,OP153,X3,NaN,3
1270,81788,OP12112,595,NaN,3


In [70]:
RealAgencyIDs = routes['agency_id']
RealAgencies = agencies[agencies['agency_id'].isin(RealAgencyIDs)]
RealAgencies.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/agency.txt", index=False)
RealAgencies

,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,agency_noc
0,OP564,Megabus,https://www.traveline.info,Europe/London,EN,0141 352 4444,MEGA
1,OP5051,FlixBus,https://www.traveline.info,Europe/London,EN,NaN,FLIX
2,OP1123,South Yorkshire Future Tram,https://www.traveline.info,Europe/London,EN,NaN,SYFT
3,OP666,North Yorkshire County Council,https://www.traveline.info,Europe/London,EN,NaN,NYCC
4,OP245,The Little White Bus,https://www.traveline.info,Europe/London,EN,NaN,UWCO
...,...,...,...,...,...,...,...
77,OP10924,Bee Network,https://www.traveline.info,Europe/London,EN,NaN,BNSM
78,OP676,Coastal and Country Coaches,https://www.traveline.info,Europe/London,EN,NaN,CACC
79,OP1554,Stringers,https://www.traveline.info,Europe/London,EN,NaN,SPMW
80,OP230,Generation Travel,https://www.traveline.info,Europe/London,EN,NaN,GTLT


In [71]:
RealTripIDs = FilteredOrderedBusLocations['trip_id']
RealTrips = trips[trips['trip_id'].isin(RealTripIDs)]
RealTrips.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/trips.txt", index=False)
RealTrips

,route_id,service_id,trip_id,trip_headsign,block_id,shape_id,wheelchair_accessible,vehicle_journey_code
3602,6554,204,VJ1161d630784ad0ddec7b4740cf923840e719ffeb,Sutton Ings,817d9c61a8c2dc2cd69ce8dad06b5882bb60c416,RPSP55584e6e5f744a61a3346ab457e2abf4085f19e2,0,VJ3506
3603,6554,204,VJ42abbb8963cf81e91083f8ac984e98e2cf6571a7,Bricknell Estate,a8cba29a2fe38cd21c1d8586c705329c5815828d,RPSP482a4c3c61652b0e474b4664af2e85ee6e5a93b1,0,VJ3507
3605,12443,205,VJd3c3856121f7e490a6f616621a1c6576d2c147aa,Thornhill Lees,NaN,NaN,0,vj_2
3607,103593,206,VJ0aa5ff6cdaf82e43b77bf5eb3366ef74b26244e3,Mixenden,0ca03de3fdbd8c3355770e07afac389c0125e8f8,RPSP4477190e9ce37215ee3ef441f55f67169462f509,0,VJ2
3656,12614,208,VJ04c79eb5880f304a8fcca45f2a60d19a1f42402a,Bradford City Centre,46d3c6dacd9a57a8b5aec3af41aa9767a14a42db,RPSPe6dd257e6517f346ca5dded89e77db322a3f25c2,0,"VJ7993,VJ8038,VJ8083,VJ8128,VJ8173"
...,...,...,...,...,...,...,...,...
58899,7965,208,VJ27b3d4776340c2706ae1584bbd1fecb68a53c1cb,Totley Brook,80e9592358d3e242f982183a11422d429f7132b1,RPSPa7b42221af75329909311b5e992eb87151d6f30e,0,"VJ7257,VJ7312,VJ7367"
58901,7965,208,VJ0ae3fed583829b7b984328f6e327e468c6c72137,Western Bank,ff2605763d6a7b0c96c3354ec753f0001f1d3a7f,RPSPc316d3ce6b2531974dfc996ff75a39d8dac11c9f,0,"VJ7290,VJ7345,VJ7400"
58904,7965,229,VJ1302769cbcb78285ae3c2d0307a60a30ab44e2b0,Totley Brook,c7ad5d021058ec38f988f7b80b2969789ab000c0,RPSP41181000ec26b65cf5dc31e77d2c7556b99cedb7,0,"VJ7289,VJ7344,VJ7399"
58906,11872,237,VJd99974661961270ba7c87bd04853e6ea6a09ae7b,Newstead,NaN,NaN,0,vj_5


In [72]:
RealShapeIDs = RealTrips['shape_id'].to_list()
RealShapes = shapes[shapes.shape_id.isin(RealShapeIDs)]
RealShapes.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/shapes.txt", index=False)

In [73]:
RealServiceIDs = RealTrips['service_id']
RealCalendar = calendar[calendar['service_id'].isin(RealServiceIDs)]
RealCalendarDates = calendar_dates[calendar_dates['service_id'].isin(RealServiceIDs)]
RealCalendar.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/calendar.txt", index=False)
RealCalendarDates.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/calendar_dates.txt", index=False)

Stop times.txt.

minimally required fields are trip_id, arrival_time and/or departure_time, stop_id, and stop_sequence (must be sequential)

In [74]:
RealTripIDs_list = RealTripIDs.to_list()
FilteredStopTimes = stop_times[stop_times.trip_id.isin(RealTripIDs_list)]
FilteredOrderedBusLocations.rename(columns={'nearest_stop_id': 'stop_id'}, inplace=True)
RealTrips2RealStopsDict = FilteredStopTimes.groupby('trip_id')['stop_id'].agg(list).to_dict()

In [75]:
RealStopTimes = FilteredStopTimes.merge(FilteredOrderedBusLocations, on=['trip_id', 'stop_id'], how='inner')
RealStopTimes['arrival_time'] = RealStopTimes['time_only']
RealStopTimes['departure_time'] = RealStopTimes['time_only']
# Filtering only columns we need to write stop_times.txt
RealStopTimes = RealStopTimes[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','stop_headsign','pickup_type','drop_off_type','shape_dist_traveled','timepoint']]
RealStopTimes.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/stop_times.txt",index=False)

Stops.txt

In [76]:
RealStopIDs = FilteredOrderedBusLocations['stop_id'].to_list()
RealStops = stops[stops.stop_id.isin(RealStopIDs)]
RealStops.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/stops.txt", index=False)

Need to write some code to automaticall zip these files.